# Benchmark CluStream vs DenStream — KDDCup99 & KDDCup98

So sánh hai thuật toán clustering dòng: **CluStream** (`river.cluster.CluStream`, Aggarwal, Han, Wang & Yu — VLDB 2003) và **DenStream** (`river.cluster.DenStream`, Cao et al. 2006) trên hai bộ dữ liệu streaming KDDCup99 và KDDCup98.

**Hai loại biểu đồ:**
1. **Cluster purity** theo cửa sổ *horizon* tại các mốc time-unit (KDDCup99).
2. **Execution time** theo *length of stream* (KDDCup99 & KDDCup98).

**Đặc trưng sử dụng:**

- **KDDCup99 — Network Intrusion Detection:** dùng đúng **34 thuộc tính liên tục** trong 42 thuộc tính (bỏ 3 cột nominal `protocol_type`/`service`/`flag`, 4 cột nhị phân `land`/`logged_in`/`is_host_login`/`is_guest_login`, và cột nhãn).
- **KDDCup98 — Charitable Donation:** dùng **56 trường** trích từ 481 trường mỗi bản ghi (bộ lịch sử quyên góp RFM: 22 `RAMNT`, 22 `RDATE`, 12 trường tổng hợp).
- Cả hai được **chuyển thành data stream theo đúng thứ tự bản ghi trong file** (data input order).

**Tham số DenStream** (theo thực nghiệm gốc): InitN=1000, v=1000, λ=0.25, ε=16, µ=10, β=0.2.
**Tham số CluStream** (theo VLDB 2003): q=50 micro-cluster (=10·k), t=2, k=5, stream speed=200, macro-clustering mỗi time unit (`time_gap=200`).

In [ ]:
from __future__ import annotations

import os
import sys

# Notebook nằm ở river/river/cluster/. Package `river` ở gốc repo (../..).
# river chưa được cài vào môi trường -> thêm gốc repo vào sys.path để import trực tiếp từ source.
_repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

In [ ]:
import inspect
import time
from collections import Counter, deque

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from river import base, cluster, preprocessing

In [14]:
# ============================================================
# Registry: tự phát hiện MỌI thuật toán clustering trong river.cluster
# ============================================================
# Duyệt river.cluster, lấy các lớp con của base.Clusterer, và trích tham số
# __init__ (tên + giá trị mặc định) để UI tự sinh ô nhập. Nhờ vậy khi river thêm
# thuật toán mới, notebook tự cập nhật danh sách mà không phải sửa tay.


def discover_clusterers():
    """Trả về dict {tên_lớp: class} cho mọi Clusterer export trong river.cluster."""
    found = {}
    for name in dir(cluster):
        if name.startswith("_"):
            continue
        obj = getattr(cluster, name)
        if inspect.isclass(obj) and issubclass(obj, base.Clusterer):
            found[name] = obj
    return dict(sorted(found.items()))


def clusterer_params(cls):
    """Trả về list (tên_param, giá_trị_mặc_định) từ __init__ của 1 Clusterer.

    Bỏ self và **kwargs. Giữ nguyên thứ tự khai báo.
    """
    sig = inspect.signature(cls.__init__)
    params = []
    for p in sig.parameters.values():
        if p.name == "self" or p.kind == p.VAR_KEYWORD:
            continue
        default = None if p.default is inspect._empty else p.default
        params.append((p.name, default))
    return params


CLUSTERERS = discover_clusterers()

# Gợi ý params "theo bài báo" cho 2 thuật toán chính (ô nhập sẽ điền sẵn giá trị này
# thay cho mặc định của river, nhưng người dùng vẫn sửa được).
PARAM_PRESETS = {
    "DenStream": dict(
        n_samples_init=1000, stream_speed=1000, decaying_factor=0.25, epsilon=16, mu=10, beta=0.2
    ),
    "CluStream": dict(
        n_macro_clusters=5,
        max_micro_clusters=50,
        micro_cluster_r_factor=2,
        time_window=1000,
        time_gap=200,
        seed=42,
    ),
}

print("Các thuật toán clustering khả dụng trong river.cluster:")
for i, name in enumerate(CLUSTERERS, 1):
    n_params = len(clusterer_params(CLUSTERERS[name]))
    print(f"  {i}. {name}  ({n_params} tham số)")

Các thuật toán clustering khả dụng trong river.cluster:
  1. CluStream  (6 tham số)
  2. DBSTREAM  (5 tham số)
  3. DenStream  (6 tham số)
  4. KMeans  (6 tham số)
  5. ODAC  (3 tham số)
  6. STREAMKMeans  (2 tham số)
  7. TextClust  (12 tham số)


In [15]:
# ============================================================
# Data loaders — trả về iterator (x: dict[str, float], y: label)
# Thứ tự streaming = thứ tự bản ghi trong file (data input order).
# ============================================================

# --- KDD99: đúng 34 thuộc tính LIÊN TỤC trong 42 thuộc tính ---
# Loại bỏ: 3 cột nominal (protocol_type, service, flag),
#          4 cột nhị phân (land, logged_in, is_host_login, is_guest_login),
#          và cột nhãn (labels)  ->  42 - 3 - 4 - 1 = 34.
KDD99_NON_CONTINUOUS = [
    "protocol_type",
    "service",
    "flag",
    "land",
    "logged_in",
    "is_host_login",
    "is_guest_login",
    "labels",
]


def iter_kddcup99(max_samples=None):
    """KDDCup99 (10% subset) tải tự động qua scikit-learn.

    Dùng đúng 34 thuộc tính liên tục (bỏ 3 nominal + 4 binary + nhãn).
    Nhãn `y` = loại kết nối (normal / các kiểu tấn công) -> dùng tính purity.
    """
    from sklearn.datasets import fetch_kddcup99

    data = fetch_kddcup99(percent10=False, shuffle=False, as_frame=True)
    df = data.frame

    y_all = df["labels"].apply(lambda v: v.decode() if isinstance(v, bytes) else str(v))

    # as_frame trả mọi cột dạng object -> ép các cột liên tục về float
    cont_cols = [c for c in df.columns if c not in KDD99_NON_CONTINUOUS]
    num_df = df[cont_cols].astype(float)
    assert num_df.shape[1] == 34, f"Kỳ vọng 34 cột liên tục, có {num_df.shape[1]}"
    feature_cols = list(num_df.columns)

    if max_samples is not None:
        num_df = num_df.iloc[:max_samples]
        y_all = y_all.iloc[:max_samples]

    for row, y in zip(num_df.itertuples(index=False, name=None), y_all):
        x = {c: float(v) for c, v in zip(feature_cols, row)}
        yield x, y


# --- KDD98: 56 trường trích từ 481 trường mỗi bản ghi (như trong [1]) ---
# [1] = Aggarwal et al. (CluStream). Bài báo không liệt kê tường minh 56 trường,
# nên ta dùng bộ 56 trường lịch sử quyên góp (RFM) thường được dùng để tái lập
# stream KDD98: 22 số tiền quyên góp lặp lại (RAMNT_3..24), 22 ngày quyên góp
# lặp lại (RDATE_3..24) và 12 trường tổng hợp về giao dịch/quyên góp.
KDD98_FIELDS = (
    [f"RAMNT_{i}" for i in range(3, 25)]  # 22 số tiền quyên góp lặp lại
    + [f"RDATE_{i}" for i in range(3, 25)]  # 22 ngày quyên góp lặp lại
    + [  # 12 trường tổng hợp
        "RAMNTALL",
        "NGIFTALL",
        "MINRAMNT",
        "MAXRAMNT",
        "LASTGIFT",
        "AVGGIFT",
        "CARDPROM",
        "NUMPROM",
        "CARDGIFT",
        "NUMPRM12",
        "TIMELAG",
        "CARDPM12",
    ]
)  # tổng = 56

# Đường dẫn file KDDCup98 (cup98LRN.txt gốc, 481 cột)
KDD98_PATH = r"C:\Users\Trieu\Downloads\kddcup98\cup98LRN.txt"


def iter_kddcup98(path=KDD98_PATH, max_samples=None):
    """KDDCup98 (cup98LRN.txt) đọc từ file local.

    Dùng 56 trường số trích từ 481 trường (bộ RFM lịch sử quyên góp).
    Nhãn `y` = TARGET_B (0/1: người nhận có quyên góp hay không).
    """
    df = pd.read_csv(path, low_memory=False)

    y_all = df["TARGET_B"].astype(int).astype(str)

    # ép về số; các ô rỗng/không phải số -> NaN -> điền 0 để stream không lỗi
    num_df = df[KDD98_FIELDS].apply(pd.to_numeric, errors="coerce").fillna(0.0)
    assert num_df.shape[1] == 56, f"Kỳ vọng 56 trường, có {num_df.shape[1]}"
    feature_cols = list(num_df.columns)

    if max_samples is not None:
        num_df = num_df.iloc[:max_samples]
        y_all = y_all.iloc[:max_samples]

    for row, y in zip(num_df.itertuples(index=False, name=None), y_all):
        x = {c: float(v) for c, v in zip(feature_cols, row)}
        yield x, y


DATASETS = {
    "KDDCup99": iter_kddcup99,
    "KDDCup98": iter_kddcup98,
}

# --- Tổng số sample mỗi dataset (đếm nhẹ, có cache) ---
# Dùng để hiển thị cho người dùng biết trần max_sample. KDD99 phải nạp frame
# (fetch có cache đĩa); KDD98 chỉ cần đếm dòng file nên rất nhanh.
_DATASET_SIZE_CACHE = {}


def dataset_size(name):
    """Trả về tổng số bản ghi của dataset (int), hoặc None nếu không xác định được."""
    if name in _DATASET_SIZE_CACHE:
        return _DATASET_SIZE_CACHE[name]

    size = None
    try:
        if name == "KDDCup98":
            with open(KDD98_PATH) as f:
                size = sum(1 for _ in f) - 1  # trừ dòng header
        elif name == "KDDCup99":
            # PHẢI khớp với iter_kddcup99 (percent10=False) — nếu để True,
            # "tổng" chỉ là subset 10% (~494k) và nút "= toàn bộ" sẽ cắt stream
            # ở ~494k thay vì hơn 4 triệu bản ghi của full dataset.
            from sklearn.datasets import fetch_kddcup99

            data = fetch_kddcup99(percent10=False, shuffle=False, as_frame=True)
            size = len(data.frame)
    except Exception:
        size = None

    _DATASET_SIZE_CACHE[name] = size
    return size

In [16]:
# ============================================================
# Purity theo cửa sổ horizon (Cao et al. 2006)
# ============================================================
# Purity của một tập điểm đã được gán cụm, theo ĐÚNG công thức trong DenStream:
#   purity = (1/K) * sum_i ( |dominant_class_in_cluster_i| / |cluster_i| )
# tức MACRO-AVERAGE: mỗi cụm đóng góp tỉ lệ dominant của riêng nó rồi chia đều
# cho số cụm K. (Khác với micro/global purity = tổng_dominant / tổng_điểm, vốn
# bị các cụm lớn chi phối — stream KDD99 rất lệch lớp nên hai cách lệch nhau rõ.)


def horizon_purity(window):
    """window: iterable các (cluster_id, true_label). Trả về purity in [0,1].

    Macro-average theo Cao et al. 2006: trung bình tỉ lệ dominant của từng cụm.
    """
    # gom nhãn thật theo từng cụm dự đoán
    clusters = {}
    for cid, label in window:
        clusters.setdefault(cid, Counter())[label] += 1

    if not clusters:
        return None

    # tỉ lệ lớp chiếm đa số trong TỪNG cụm, rồi lấy trung bình trên K cụm
    ratios = [max(counter.values()) / sum(counter.values()) for counter in clusters.values()]
    return sum(ratios) / len(ratios)

In [17]:
# ============================================================
# Helpers dùng chung cho phần chạy benchmark
# ============================================================
def _is_ready(model):
    """Mô hình đã sẵn sàng dự đoán chưa (DenStream .initialized / CluStream._initialized)."""
    if hasattr(model, "initialized"):
        return model.initialized
    return getattr(model, "_initialized", True)


def build_model(cls_name, params):
    """Khởi tạo 1 clusterer từ tên lớp + dict params (ép kiểu theo mặc định)."""
    return CLUSTERERS[cls_name](**params)


# --- Chạy purity theo TIME UNIT trên 1 dataset ---
def run_purity_by_time_unit(
    make_model, stream_iter, stream_speed, horizon_units, max_samples, use_scaler=False
):
    """Trả về dict {time_unit: purity} cho 1 mô hình trên 1 stream.

    Purity được tính bằng cách CHỤP cụm 1 lần tại mỗi mốc time-unit rồi gán nhãn
    cho toàn bộ điểm trong cửa sổ horizon bằng đúng snapshot đó. Cách này tránh
    lỗi cluster-id không ổn định: DenStream/CluStream đánh số lại cụm ở mỗi lần
    `predict_one`, nên KHÔNG được trộn cid lấy từ các lần gọi ở các time-unit
    khác nhau vào cùng một cửa sổ.

    `use_scaler`: bật StandardScaler (mặc định). LƯU Ý tính nhất quán tham số —
    preset "theo bài báo" (epsilon=16) dùng cho dữ liệu KHÔNG chuẩn hoá; nếu bật
    scaler thì phải hạ epsilon về thang đã chuẩn hoá, ngược lại một micro-cluster
    khổng lồ sẽ nuốt gần hết điểm và purity cao giả tạo.
    """
    model = make_model()
    scaler = preprocessing.StandardScaler() if use_scaler else None
    # cửa sổ giữ (x_đã_xử_lý, nhãn_thật) của các điểm gần nhất
    window = deque(maxlen=stream_speed * horizon_units)
    purity_by_tu = {}

    n = 0
    for x, y in stream_iter:
        n += 1
        if scaler is not None:
            scaler.learn_one(x)
            x_proc = scaler.transform_one(x)
        else:
            x_proc = x
        model.learn_one(x_proc)
        window.append((x_proc, y))

        if n % stream_speed == 0 and _is_ready(model):
            # CHỤP cụm 1 lần: model đứng yên trong vòng lặp này nên mọi
            # predict_one dùng chung một kết quả phân cụm -> cid nhất quán.
            labelled = ((model.predict_one(x_proc), y_true) for x_proc, y_true in window)
            p = horizon_purity(labelled)
            if p is not None:
                purity_by_tu[n // stream_speed] = p

        if max_samples is not None and n >= max_samples:
            break
    return purity_by_tu


# --- Chạy đo Execution time theo LENGTH OF STREAM trên 1 dataset ---
def run_exec_time(make_model, stream_iter, rec_every, max_samples, use_scaler=True):
    """Trả về (n_points_list, elapsed_list): thời gian tích luỹ theo độ dài stream."""
    model = make_model()
    scaler = preprocessing.StandardScaler() if use_scaler else None
    n_points, elapsed = [], []

    n = 0
    t0 = time.perf_counter()
    for x, y in stream_iter:
        n += 1
        if scaler is not None:
            scaler.learn_one(x)
            x_proc = scaler.transform_one(x)
        else:
            x_proc = x
        model.learn_one(x_proc)
        if _is_ready(model):
            model.predict_one(x_proc)

        if n % rec_every == 0:
            n_points.append(n)
            elapsed.append(time.perf_counter() - t0)

        if max_samples is not None and n >= max_samples:
            break
    return n_points, elapsed

---
# 🎛️ Interactive benchmark wizard

Chạy cell dưới đây để mở giao diện tương tác. Luồng thao tác:

1. **Chọn thuật toán** — liệt kê mọi clusterer trong `river.cluster`, chọn theo số rồi bấm *Chọn & nhập params*.
2. **Nhập params** — ô nhập tự sinh theo `__init__` của thuật toán (điền sẵn preset nếu có). Bấm *Lưu* để quay lại màn hình chọn.
3. Màn hình chọn có nút **➕ Thêm thuật toán nữa** và **▶️ Bắt đầu chạy** (chạy khi đã thêm ≥1 thuật toán).
4. **Cấu hình chạy** — chọn dataset (KDDCup98 / KDDCup99, có thể nhiều bộ), nhập **max sample cho từng mô hình**, và thêm các **kịch bản (bộ 4)**: mỗi kịch bản gồm `horizon` + `stream speed` + mốc *time-unit* (purity) + mốc *length-of-stream* (exec time) — các mốc đi liền theo từng bộ, không dùng chung.
5. Bấm **🚀 Chạy benchmark** — với mỗi dataset × mỗi kịch bản, notebook vẽ 1 đồ thị purity + 1 đồ thị execution time ngay bên dưới.

> Không thích UI? Vẫn có thể gọi trực tiếp `run_purity_by_time_unit(...)` / `run_exec_time(...)` như API thường.

In [18]:
# ============================================================
# State + tiện ích ép kiểu params cho wizard
# ============================================================
# WIZARD_STATE giữ danh sách thuật toán người dùng đã cấu hình, mỗi phần tử:
#   {"name": <tên hiển thị>, "cls": <tên lớp>, "params": {..}}
# Cho phép thêm cùng 1 lớp nhiều lần với params khác nhau (đặt hậu tố #2, #3...).

WIZARD_STATE = {
    "models": [],  # list các dict thuật toán đã lưu
    "run_config": {},  # datasets, scenarios (bộ 4), max_samples (dict/mô hình)
}


def _coerce(value_str, default):
    """Ép chuỗi người dùng nhập về kiểu theo giá trị mặc định của param."""
    s = value_str.strip()
    if s == "" or s.lower() == "none":
        return None
    if isinstance(default, bool):
        return s.lower() in ("1", "true", "yes", "y", "t")
    if isinstance(default, int) and not isinstance(default, bool):
        try:
            return int(s)
        except ValueError:
            return float(s)
    if isinstance(default, float):
        return float(s)
    # default là None hoặc str -> thử số trước, không được thì giữ chuỗi
    try:
        return int(s)
    except ValueError:
        try:
            return float(s)
        except ValueError:
            return s


def _display_name(cls_name):
    """Sinh tên hiển thị duy nhất (CluStream, CluStream #2, ...)."""
    existing = [m["name"] for m in WIZARD_STATE["models"]]
    if cls_name not in existing:
        return cls_name
    k = 2
    while f"{cls_name} #{k}" in existing:
        k += 1
    return f"{cls_name} #{k}"

In [19]:
# ============================================================
# WIZARD UI — các màn hình dùng ipywidgets
# ============================================================
# Kiến trúc: 1 vùng hiển thị chính `SCREEN` (VBox). Mỗi hàm show_*_screen()
# dựng lại nội dung SCREEN.children cho từng bước. `RESULT_OUT` là vùng in
# kết quả benchmark (đồ thị) tách riêng để không bị xoá khi chuyển màn hình.

SCREEN = widgets.VBox()
RESULT_OUT = widgets.Output()


# ---------- Bước 1: chọn thuật toán ----------
def show_select_screen():
    title = widgets.HTML("<h3>Bước 1 — Chọn thuật toán</h3>")

    options = [(f"{i}. {name}", name) for i, name in enumerate(CLUSTERERS, 1)]
    dd = widgets.Dropdown(
        options=options,
        description="Thuật toán:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="400px"),
    )

    btn_pick = widgets.Button(description="Chọn & nhập params", button_style="info", icon="sliders")
    btn_pick.on_click(lambda _: show_param_screen(dd.value))

    # danh sách thuật toán đã thêm
    if WIZARD_STATE["models"]:
        rows = "".join(
            f"<li><b>{m['name']}</b>: "
            f"{', '.join(f'{k}={v}' for k, v in m['params'].items()) or '(mặc định)'}</li>"
            for m in WIZARD_STATE["models"]
        )
        added_html = widgets.HTML(f"<b>Đã thêm ({len(WIZARD_STATE['models'])}):</b><ul>{rows}</ul>")
    else:
        added_html = widgets.HTML("<i>Chưa thêm thuật toán nào.</i>")

    btn_add = widgets.Button(description="➕ Thêm thuật toán nữa", icon="plus")
    btn_add.on_click(lambda _: show_select_screen())  # ở lại màn chọn để chọn tiếp

    btn_clear = widgets.Button(description="🗑️ Xoá hết", button_style="warning")

    def _clear(_):
        WIZARD_STATE["models"].clear()
        show_select_screen()

    btn_clear.on_click(_clear)

    btn_run = widgets.Button(
        description="▶️ Bắt đầu chạy",
        button_style="success",
        icon="play",
        disabled=len(WIZARD_STATE["models"]) == 0,
    )
    btn_run.on_click(lambda _: show_config_screen())

    controls = widgets.HBox([dd, btn_pick])
    nav = widgets.HBox([btn_add, btn_run, btn_clear])
    SCREEN.children = [title, controls, added_html, nav]

In [20]:
# ---------- Bước 2: nhập params cho thuật toán vừa chọn ----------
def show_param_screen(cls_name):
    cls = CLUSTERERS[cls_name]
    title = widgets.HTML(f"<h3>Bước 2 — Nhập tham số cho <code>{cls_name}</code></h3>")

    preset = PARAM_PRESETS.get(cls_name, {})
    param_defs = clusterer_params(cls)  # [(tên, mặc_định), ...]

    inputs = {}
    rows = []
    for pname, pdefault in param_defs:
        init_val = preset.get(pname, pdefault)
        txt = widgets.Text(
            value="" if init_val is None else str(init_val),
            description=pname,
            style={"description_width": "160px"},
            layout=widgets.Layout(width="360px"),
        )
        hint = widgets.HTML(f"<span style='color:gray'>mặc định river = {pdefault!r}</span>")
        inputs[pname] = (txt, pdefault)
        rows.append(widgets.HBox([txt, hint]))

    if not param_defs:
        rows.append(widgets.HTML("<i>Thuật toán này không có tham số cấu hình.</i>"))

    btn_save = widgets.Button(description="💾 Lưu & quay lại", button_style="success", icon="check")

    def _save(_):
        params = {}
        for pname, (txt, pdefault) in inputs.items():
            val = _coerce(txt.value, pdefault)
            # bỏ qua param để trống mà mặc định cũng None (không truyền -> dùng default)
            if val is None and pdefault is None and txt.value.strip() == "":
                continue
            params[pname] = val
        WIZARD_STATE["models"].append(
            {
                "name": _display_name(cls_name),
                "cls": cls_name,
                "params": params,
            }
        )
        show_select_screen()

    btn_save.on_click(_save)

    btn_cancel = widgets.Button(description="↩️ Huỷ", icon="times")
    btn_cancel.on_click(lambda _: show_select_screen())

    SCREEN.children = [title, widgets.VBox(rows), widgets.HBox([btn_save, btn_cancel])]

In [21]:
# ---------- Bước 3: cấu hình chạy — mỗi KỊCH BẢN tự chứa mọi thứ ----------
# Mỗi kịch bản = 1 bộ độc lập:
#   dataset · max sample · horizon · stream speed · chuẩn hoá (scaler)
#   · CHỌN loại đồ thị (purity / exec-time) -> chỉ khi chọn mới hiện ô nhập mốc.
# Nhờ vậy chỉ chạy & vẽ đúng những gì bạn cần.
_BTN_AUTO = widgets.Layout(width="auto")


def show_config_screen():
    title = widgets.HTML("<h3>Bước 3 — Cấu hình các kịch bản</h3>")
    hint = widgets.HTML(
        "<span style='color:gray'>Mỗi kịch bản là 1 bộ độc lập. Tích loại đồ thị "
        "muốn vẽ; ô nhập mốc chỉ hiện khi đồ thị tương ứng được chọn.</span>"
    )

    sc_container = widgets.VBox()
    sc_rows = []  # list dict widget của từng kịch bản

    def _refresh():
        for idx, e in enumerate(sc_rows, 1):
            e["box"].children[0].value = f"<b>🎯 Kịch bản #{idx}</b>"
        sc_container.children = [e["box"] for e in sc_rows]

    def _add_scenario(
        ds=None, mx=94000, h=1, s=200, tu="43, 51, 86, 370", ln="20000, 40000, 60000, 80000"
    ):
        ds = ds or next(iter(DATASETS))
        header = widgets.HTML()

        w_ds = widgets.Dropdown(
            options=list(DATASETS),
            value=ds,
            description="dataset:",
            style={"description_width": "70px"},
            layout=widgets.Layout(width="240px"),
        )
        w_max = widgets.IntText(
            value=mx,
            description="max sample:",
            style={"description_width": "90px"},
            layout=widgets.Layout(width="230px"),
        )
        btn_full = widgets.Button(description="= toàn bộ", layout=_BTN_AUTO)
        ds_info = widgets.HTML()

        def _show_size(*_):
            size = dataset_size(w_ds.value)
            ds_info.value = (
                f"<span style='color:gray'>tổng: <b>{size:,}</b> sample</span>"
                if size is not None
                else "<span style='color:gray'>tổng: ?</span>"
            )

        def _use_full(_):
            size = dataset_size(w_ds.value)
            if size is not None:
                w_max.value = size

        w_ds.observe(_show_size, names="value")
        btn_full.on_click(_use_full)
        _show_size()

        w_h = widgets.IntText(
            value=h,
            description="horizon:",
            style={"description_width": "70px"},
            layout=widgets.Layout(width="170px"),
        )
        w_s = widgets.IntText(
            value=s,
            description="speed:",
            style={"description_width": "70px"},
            layout=widgets.Layout(width="170px"),
        )

        # StandardScaler: mặc định BẬT. Nếu chạy preset "theo bài báo" (epsilon=16)
        # thì nên TẮT, vì ε=16 dành cho dữ liệu chưa chuẩn hoá — bật scaler sẽ khiến
        # một micro-cluster khổng lồ nuốt gần hết điểm, purity cao giả tạo.
        cb_scaler = widgets.Checkbox(
            value=True,
            description="Chuẩn hoá (StandardScaler)",
            indent=False,
            layout=widgets.Layout(width="260px"),
        )

        # --- chọn loại đồ thị + ô nhập mốc (ẩn/hiện theo checkbox) ---
        cb_purity = widgets.Checkbox(
            value=True,
            description="Vẽ CLUSTER PURITY",
            indent=False,
            layout=widgets.Layout(width="220px"),
        )
        cb_exec = widgets.Checkbox(
            value=True,
            description="Vẽ EXECUTION TIME",
            indent=False,
            layout=widgets.Layout(width="220px"),
        )
        w_tu = widgets.Text(
            value=tu,
            description="mốc TIME-UNIT (purity):",
            style={"description_width": "180px"},
            layout=widgets.Layout(width="520px"),
        )
        w_len = widgets.Text(
            value=ln,
            description="mốc LENGTH (exec time):",
            style={"description_width": "180px"},
            layout=widgets.Layout(width="520px"),
        )

        def _toggle(*_):
            w_tu.layout.display = "" if cb_purity.value else "none"
            w_len.layout.display = "" if cb_exec.value else "none"

        cb_purity.observe(_toggle, names="value")
        cb_exec.observe(_toggle, names="value")
        _toggle()

        btn_del = widgets.Button(
            description="✕ Xoá kịch bản này", button_style="danger", layout=_BTN_AUTO
        )

        box = widgets.VBox(
            [
                header,
                widgets.HBox([w_ds, ds_info]),
                widgets.HBox([w_max, btn_full]),
                widgets.HBox([w_h, w_s]),
                cb_scaler,
                widgets.HBox([cb_purity, cb_exec]),
                w_tu,
                w_len,
                btn_del,
            ],
            layout=widgets.Layout(border="1px solid #ccc", padding="8px", margin="6px 0"),
        )
        entry = {
            "box": box,
            "ds": w_ds,
            "max": w_max,
            "h": w_h,
            "s": w_s,
            "tu": w_tu,
            "len": w_len,
            "pp": cb_purity,
            "pe": cb_exec,
            "sc": cb_scaler,
        }
        sc_rows.append(entry)

        def _del(_):
            if len(sc_rows) > 1:  # luôn giữ ít nhất 1 kịch bản
                sc_rows.remove(entry)
                _refresh()

        btn_del.on_click(_del)
        _refresh()

    _add_scenario()  # kịch bản mặc định

    btn_add_sc = widgets.Button(description="➕ Thêm kịch bản", icon="plus", layout=_BTN_AUTO)
    btn_add_sc.on_click(lambda _: _add_scenario())

    err = widgets.HTML()
    btn_run = widgets.Button(
        description="🚀 Chạy benchmark", button_style="success", icon="rocket", layout=_BTN_AUTO
    )

    def _run(_):
        scenarios, seen = [], set()
        for e in sc_rows:
            plot_purity, plot_exec = e["pp"].value, e["pe"].value
            if not (plot_purity or plot_exec):
                continue  # kịch bản không vẽ gì -> bỏ
            tu = _parse_marks(e["tu"].value) if plot_purity else []
            ln = _parse_marks(e["len"].value) if plot_exec else []
            key = (
                e["ds"].value,
                e["max"].value,
                e["h"].value,
                e["s"].value,
                e["sc"].value,
                plot_purity,
                plot_exec,
                tuple(tu),
                tuple(ln),
            )
            if key in seen:  # bỏ kịch bản trùng hoàn toàn
                continue
            seen.add(key)
            scenarios.append(
                {
                    "dataset": e["ds"].value,
                    "max_samples": e["max"].value,
                    "horizon": e["h"].value,
                    "speed": e["s"].value,
                    "use_scaler": e["sc"].value,
                    "plot_purity": plot_purity,
                    "plot_exec": plot_exec,
                    "tu_marks": tu,
                    "len_marks": ln,
                }
            )
        if not scenarios:
            err.value = (
                "<span style='color:red'>⚠️ Cần ít nhất 1 kịch bản có chọn loại đồ thị.</span>"
            )
            return
        WIZARD_STATE["run_config"] = {"scenarios": scenarios}
        run_wizard_benchmark()

    btn_run.on_click(_run)

    btn_back = widgets.Button(
        description="↩️ Quay lại chọn thuật toán", icon="arrow-left", layout=_BTN_AUTO
    )
    btn_back.on_click(lambda _: show_select_screen())

    SCREEN.children = [
        title,
        hint,
        sc_container,
        btn_add_sc,
        widgets.HTML("<hr>"),
        err,
        widgets.HBox([btn_back, btn_run]),
    ]

In [22]:
# ---------- Tiện ích: parse chuỗi mốc ----------
# (Bước "chọn mốc" đã được gộp thẳng vào Bước 3 — mỗi kịch bản mang theo
#  mốc time-unit & mốc length riêng, nên không còn màn hình chọn mốc tách rời.)
def _parse_marks(text):
    """'43, 51, 86' -> [43, 51, 86]. Bỏ phần tử rỗng/không hợp lệ."""
    out = []
    for tok in text.replace(";", ",").split(","):
        tok = tok.strip()
        if not tok:
            continue
        try:
            out.append(int(float(tok)))
        except ValueError:
            pass
    return out

In [23]:
# ============================================================
# Bước 4: chạy benchmark theo WIZARD_STATE + vẽ đồ thị
# ============================================================
# Mỗi KỊCH BẢN tự chứa: dataset · max_samples · horizon · speed · use_scaler
# · cờ vẽ purity/exec · mốc tương ứng. Chỉ vẽ đúng loại đồ thị được chọn.
#
# LƯU Ý double-render: với backend `%matplotlib inline`, mọi figure còn "mở"
# sẽ bị flush hiển thị lần nữa khi cell/khối kết thúc. Vì thế PHẢI đóng figure
# ngay sau khi hiển thị. Ở đây ta hiển thị thủ công bằng IPython.display rồi
# `plt.close(fig)` — tránh mọi khả năng vẽ lại lần 2.
_PALETTE = plt.rcParams["axes.prop_cycle"].by_key()["color"]


def _model_color(i):
    return _PALETTE[i % len(_PALETTE)]


def _show_and_close(fig):
    """Hiển thị figure đúng 1 lần rồi đóng (chống inline flush vẽ lại)."""
    display(fig)
    plt.close(fig)


def _plot_purity(results_tu, marks, title):
    names = list(results_tu.keys())
    x = np.arange(len(marks))
    width = 0.8 / max(len(names), 1)

    fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(marks)), 5))
    for mi, name in enumerate(names):
        u2p = results_tu[name]
        vals = [100 * u2p.get(m, 0.0) for m in marks]
        bars = ax.bar(
            x + mi * width, vals, width, label=name, color=_model_color(mi), edgecolor="black"
        )
        ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)

    ax.set_xticks(x + width * (len(names) - 1) / 2)
    ax.set_xticklabels(marks)
    ax.set_xlabel("Stream (in time units)")
    ax.set_ylabel("Cluster Purity %")
    ax.set_title(title)
    ax.set_ylim(0, 105)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    _show_and_close(fig)


def _plot_exec_time(results_et, length_marks, title):
    names = list(results_et.keys())
    max_elapsed = max((el[-1] for _, el in results_et.values() if el), default=1.0)
    y_top = len(length_marks) if length_marks else 8
    scale = y_top / max_elapsed if max_elapsed else 1.0

    fig, ax = plt.subplots(figsize=(8, 5))
    markers = ["o", "s", "^", "D", "v", "*", "P", "X"]
    for mi, name in enumerate(names):
        npts, el = results_et[name]
        if npts:
            ax.plot(
                npts,
                [e * scale for e in el],
                label=name,
                color=_model_color(mi),
                marker=markers[mi % len(markers)],
                ms=4,
                lw=1.5,
                markevery=0.1,
            )

    ax.set_xlabel("Length of Stream")
    ax.set_ylabel("Execution Time (normalized)")
    ax.set_title(title)
    if length_marks:
        ax.set_xticks(length_marks)
        ax.set_xticklabels([f"{int(t / 1000)}k" if t >= 1000 else str(t) for t in length_marks])
        ax.set_xlim(0, max(length_marks) * 1.05)
    ax.set_yticks(range(0, y_top + 1))
    ax.set_ylim(0, y_top + 0.5)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    _show_and_close(fig)
    print(
        f"[{title}] scale = {scale:.4e}  "
        f"(thời gian thực chậm nhất = {max_elapsed:.1f}s -> {y_top} đơn vị)\n"
    )


def run_wizard_benchmark():
    cfg = WIZARD_STATE["run_config"]
    models = WIZARD_STATE["models"]

    SCREEN.children = [
        widgets.HTML(
            "<h3>Đang chạy benchmark ...</h3>"
            "<span style='color:gray'>Xem tiến trình & kết quả bên dưới.</span>"
        )
    ]

    RESULT_OUT.clear_output(wait=True)
    with RESULT_OUT:
        for si, sc in enumerate(cfg["scenarios"], 1):
            ds_name = sc["dataset"]
            cap = sc["max_samples"]
            h, v = sc["horizon"], sc["speed"]
            use_scaler = sc.get("use_scaler", True)
            loader = DATASETS[ds_name]
            scaler_tag = "scaler=on" if use_scaler else "scaler=off"

            print(
                f"\n{'=' * 60}\nKịch bản #{si}: {ds_name} · max={cap:,} · "
                f"horizon={h} · speed={v} · {scaler_tag}\n{'=' * 60}",
                flush=True,
            )

            # --- PURITY (chỉ khi được chọn) ---
            if sc["plot_purity"]:
                purity_res = {}
                for m in models:
                    mk = (lambda mm: lambda: build_model(mm["cls"], mm["params"]))(m)
                    print(
                        f"[purity] {ds_name} h={h} v={v} {scaler_tag} :: {m['name']} ...",
                        flush=True,
                    )
                    purity_res[m["name"]] = run_purity_by_time_unit(
                        mk,
                        loader(max_samples=cap),
                        stream_speed=v,
                        horizon_units=h,
                        max_samples=cap,
                        use_scaler=use_scaler,
                    )
                _plot_purity(
                    purity_res,
                    sc["tu_marks"],
                    f"Cluster purity — {ds_name} · Kịch bản #{si}\n"
                    f"horizon={h}, stream speed={v}, {scaler_tag}",
                )

            # --- EXEC TIME (chỉ khi được chọn) ---
            if sc["plot_exec"]:
                len_cap = max(cap, max(sc["len_marks"], default=0))
                et_res = {}
                for m in models:
                    mk = (lambda mm: lambda: build_model(mm["cls"], mm["params"]))(m)
                    print(
                        f"[exec ] {ds_name} {scaler_tag} :: {m['name']} (tới {len_cap:,} điểm) ...",
                        flush=True,
                    )
                    et_res[m["name"]] = run_exec_time(
                        mk,
                        loader(max_samples=len_cap),
                        rec_every=max(1, len_cap // 100),
                        max_samples=len_cap,
                        use_scaler=use_scaler,
                    )
                _plot_exec_time(
                    et_res,
                    sc["len_marks"],
                    f"Execution time vs. length of stream — {ds_name} · Kịch bản #{si}\n"
                    f"horizon={h}, stream speed={v}, {scaler_tag}",
                )

        print("\n✅ Hoàn tất tất cả kịch bản.", flush=True)

    datasets_used = sorted({sc["dataset"] for sc in cfg["scenarios"]})
    btn_restart = widgets.Button(
        description="🔄 Cấu hình lần chạy mới",
        button_style="info",
        icon="redo",
        layout=widgets.Layout(width="auto"),
    )
    btn_restart.on_click(lambda _: show_select_screen())
    SCREEN.children = [
        widgets.HTML(
            f"<h3>✅ Hoàn tất {len(cfg['scenarios'])} kịch bản ({', '.join(datasets_used)})</h3>"
        ),
        btn_restart,
    ]

---
## ▶️ Khởi động wizard

Chạy cell dưới để mở giao diện. Mọi thao tác chọn thuật toán → nhập params → cấu hình → chọn mốc → chạy đều nằm trong khung widget bên dưới; kết quả (2 đồ thị purity + execution time) hiện trong vùng output ngay sau đó.

In [ ]:
# ============================================================
# Khởi động wizard: hiển thị màn hình đầu tiên + vùng kết quả
# ============================================================
show_select_screen()
display(SCREEN)
display(RESULT_OUT)

Output()